# Week 4 Baseline First — 高空紅外線無人機影像車輛偵測

學號：7114029015　姓名：黃柏瑜

本 Notebook 在 **同一份 Dataset（`hituav-vehicle-v0.2`）、同一個 W3 Split、同一套 Metric 與同一個 Information Boundary** 下比較：

| Role | Method |
| --- | --- |
| Baseline 1（W2 current / naive） | 全域 Otsu 門檻 + 連通元件（熱源 = 亮像素） |
| Baseline 2（credible simple） | 雙極性局部對比候選框 + HOG/幾何特徵 + 簡單分類器（LR / Gradient Boosting，於 Val 選擇） |
| Candidate | YOLO11n（COCO 預訓練，於 Train 微調） |

所有方法只吃單張灰階 IR 影格像素；高度、角度、日期、檔名都不進模型。

## 0. 設定

In [1]:
from pathlib import Path
import sys, time, json
import numpy as np
import pandas as pd
import cv2

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
import week04_baselines as wb

DATA_DIR = ROOT / "data" / "hit_uav"                  # W3 dataset location
DATASET_VERSION = "hituav-vehicle-v0.2"
TARGET = "vehicle (Car + OtherVehicle)"               # W3 vehicle_map_v0.2
FEATURES = "single 640x512 grayscale IR frame only"  # deployment-time input
RANDOM_STATE = 42
YOLO_WEIGHTS = ROOT / "models" / "week04_yolo11n_hituav_v02.pt"
TRAIN_YOLO = False   # True = 重新訓練（CPU 約 1.5–2 小時）；False = 使用已訓練好的權重
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Repo root:", ROOT)

Repo root: /home/claude/repo


## 1. Task / Dataset / Split（沿用 W3，原封不動）

In [2]:
im, an = wb.load_dataset(DATA_DIR)
n_dl = wb.ensure_images(im, DATA_DIR)
print("downloaded missing images:", n_dl)
veh = an[an.category_id.isin(wb.VEH)].copy()
dc = an[an.category_id == wb.DONTCARE].copy()
veh["scale"] = pd.cut(veh.w * veh.h, [0, 16**2, 32**2, np.inf], labels=["tiny", "small", "medium+"], right=False)
veh = veh.merge(im[["id", "filename", "split", "daynight", "altitude_m", "angle_deg", "date_captured"]], left_on="image_id", right_on="id", suffixes=("", "_img"))

split_rows = pd.DataFrame({
    "frames": im.groupby("split").size(),
    "vehicle_boxes": veh.groupby("split").size(),
    "dates": im.groupby("split").date_captured.apply(lambda s: ", ".join(sorted(s.unique()))),
}).reindex(["train", "val", "test"])
print(split_rows)
g = {s: set(im.loc[im.split == s, "flight_group"]) for s in ["train", "val", "test"]}
print("flight_group overlap train∩test:", len(g["train"] & g["test"]), "| train∩val:", len(g["train"] & g["val"]))
print("Test vehicle scale:", veh[veh.split == "test"].scale.value_counts().to_dict())

downloaded missing images: 0
       frames  vehicle_boxes                                              dates
split                                                                          
train    1954           3707  20201217, 20210114, 20210115, 20210116, 202101...
val       316           1263                                           20210119
test      628           2489                                 20210121, 20210123
flight_group overlap train∩test: 0 | train∩val: 0
Test vehicle scale: {'medium+': 1802, 'small': 674, 'tiny': 13}


### 1a. Data check：車輛是「熱」的嗎？（決定 Baseline 設計）
比較每個車輛框內平均灰階與外圍一圈背景。若車比背景暗（冷），只抓亮像素的 Otsu 一定抓不到。

In [3]:
def polarity(df):
    out = []
    for f, grp in df.groupby("filename"):
        img = cv2.imread(str(DATA_DIR / "images" / f), 0).astype(float)
        for a in grp.itertuples():
            x, y, w, h = a.x, a.y, a.w, a.h; px, py = w // 2, h // 2
            inner = img[y:y+h, x:x+w]; outer = img[max(0, y-py):y+h+py, max(0, x-px):x+w+px]
            ring = (outer.sum() - inner.sum()) / max(1, outer.size - inner.size)
            out.append(inner.mean() - ring)
    return pd.Series(out, index=[i for _, grp in df.groupby("filename") for i in grp.index])
veh["contrast"] = polarity(veh)
veh["cold"] = veh.contrast < 0
print("vehicles darker than surroundings (all):  %.1f%%" % (100 * veh.cold.mean()))
print(veh.groupby("daynight").cold.mean().mul(100).round(1).rename("cold % by day/night").to_string())
print(veh.groupby("split").cold.mean().mul(100).round(1).rename("cold % by split").to_string())

vehicles darker than surroundings (all):  70.6%
daynight
day      43.1
night    84.9
split
test     57.7
train    75.6
val      81.5


## 2. Fair Comparison Contract
- Dataset version = `hituav-vehicle-v0.2`（HIT-UAV normal_json，Car+OtherVehicle → vehicle）
- Split = Group/Time split by `date_captured`；Boundary = Train ≤ 2021-01-20（6 天）、Val = 2021-01-19、Test = 2021-01-21 + 2021-01-23；Seed = 42（YOLO 訓練／負樣本抽樣）
- Baseline 1 = Global Otsu + CCA（hot pixels）
- Baseline 2 = Bi-polar local-contrast proposals + HOG/geometry + classifier（LR 與 GB 於 Val 擇一）
- Primary Metric = vehicle AP@0.5（Test，↑）
- Failure Metric = vehicle Miss Rate（FN / GT，於 Val 選定門檻，↓）；輔助：False alarms per 100 frames（↓）
- Threshold policy = 每個方法都在 Val 上取 F1 最大的 score 門檻，Test 鎖定不調
- Information boundary = 只用單張灰階影格像素；不使用高度、角度、日期、檔名；DontCare 只在評估時忽略
- Runtime / Cost = 同一台 CPU（2 cores）單張平均 ms/frame（含前處理，不含讀檔）

In [4]:
SIZE = veh[veh.split == "train"]
AMIN, AMAX = np.percentile(SIZE.w * SIZE.h, 1), np.percentile(SIZE.w * SIZE.h, 99)
ARMAX = np.percentile(np.maximum(SIZE.w, SIZE.h) / np.minimum(SIZE.w, SIZE.h), 99)
size_rule = lambda w, h: AMIN * 0.5 <= w * h <= AMAX * 1.5 and max(w, h) / max(1, min(w, h)) <= ARMAX * 1.2
print(f"size rule (from Train GT only): area {AMIN:.0f}–{AMAX:.0f} px², aspect ≤ {ARMAX:.2f}")
ids = {s: list(im[im.split == s].id) for s in ["train", "val", "test"]}
fname = dict(zip(im.id, im.filename))
def gray(iid): return cv2.imread(str(DATA_DIR / "images" / fname[iid]), 0)
preds, runtime = {}, {}

size rule (from Train GT only): area 143–18635 px², aspect ≤ 3.17


## 3. Baseline 1｜Global Otsu + CCA

In [5]:
for s in ["val", "test"]:
    rows, t_total = [], 0.0
    for iid in ids[s]:
        g = gray(iid); t0 = time.perf_counter()
        dets = wb.otsu_cca(g, size_rule)
        t_total += time.perf_counter() - t0
        rows += [dict(d, image_id=iid) for d in dets]
    preds[("B1 Otsu+CCA", s)] = pd.DataFrame(rows)
    runtime[("B1 Otsu+CCA", s)] = 1000 * t_total / len(ids[s])
print({k: f"{len(v)} boxes" for k, v in preds.items()}, {k: f"{v:.1f} ms" for k, v in runtime.items()})

{('B1 Otsu+CCA', 'val'): '10038 boxes', ('B1 Otsu+CCA', 'test'): '14463 boxes'} {('B1 Otsu+CCA', 'val'): '6.4 ms', ('B1 Otsu+CCA', 'test'): '6.1 ms'}


## 4. Baseline 2｜Bi-polar local contrast proposals + HOG + classifier
候選框：`|I − blur(I)|` 亮、暗兩個方向都抓；分類器只用 Train 的候選框訓練（IoU ≥ 0.5 為正樣本，負樣本抽 15%，seed 42）。

In [6]:
PROP_CFG = [(1.0, 61, 7), (0.8, 101, 7)]   # (k·σ threshold, background kernel, closing kernel) — chosen on Val proposal recall
def propose(iid):
    g = gray(iid); t0 = time.perf_counter(); ps = []
    for k, b, c in PROP_CFG:
        ps += wb.local_contrast_proposals(g, size_rule, k, b, c)
    return g, ps, time.perf_counter() - t0

# training proposals and labels
rows = []
for iid in ids["train"]:
    _, ps, _ = propose(iid); rows += [dict(p, image_id=iid) for p in ps]
Ptr = pd.DataFrame(rows)
pm, _ = wb.match(Ptr.assign(score=0.0), veh, dc, ids["train"])
Ptr["label"] = (pm.loc[Ptr.index, "best_iou"] >= 0.5).astype(int)
Ptr = Ptr[~(pm.loc[Ptr.index, "ignored"] & (Ptr.label == 0))]
Ptr = pd.concat([Ptr[Ptr.label == 1], Ptr[Ptr.label == 0].sample(frac=0.15, random_state=RANDOM_STATE)])
Xtr, ytr = [], []
for iid, grp in Ptr.groupby("image_id"):
    g = gray(iid)
    Xtr += [wb.proposal_features(g, p) for p in grp.to_dict("records")]; ytr += list(grp.label)
Xtr, ytr = np.vstack(Xtr), np.array(ytr)
print("train proposals:", len(ytr), "positives:", int(ytr.sum()))

train proposals: 54156 positives: 4888


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

clfs = {
    "B2a HOG+LR": make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced")),
    "B2b HOG+GB": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, class_weight="balanced", random_state=RANDOM_STATE),
}
for name, clf in clfs.items():
    t0 = time.perf_counter(); clf.fit(Xtr, ytr); print(name, "fit seconds:", round(time.perf_counter() - t0, 1))

def nms(df, thr=0.45):
    keep = []
    for _, g in df.groupby("image_id"):
        g = g.sort_values("score", ascending=False); b = g[["x", "y", "w", "h"]].values.astype(float); alive = np.ones(len(g), bool)
        for i in range(len(g)):
            if alive[i]:
                keep.append(g.index[i]); alive &= wb.iou_matrix(b[i:i+1], b)[0] < 0.45
    return df.loc[keep]

for s in ["val", "test"]:
    rows, feats, t_prop = [], [], 0.0
    for iid in ids[s]:
        g, ps, dt = propose(iid); t0 = time.perf_counter()
        feats += [wb.proposal_features(g, p) for p in ps]; t_prop += dt + time.perf_counter() - t0
        rows += [dict(p, image_id=iid) for p in ps]
    P, X = pd.DataFrame(rows), np.vstack(feats)
    for name, clf in clfs.items():
        t0 = time.perf_counter(); P2 = P.copy(); P2["score"] = clf.predict_proba(X)[:, 1]; P2 = nms(P2)
        preds[(name, s)] = P2; runtime[(name, s)] = 1000 * (t_prop + time.perf_counter() - t0) / len(ids[s])
print({k: f"{v:.1f} ms" for k, v in runtime.items()})

B2a HOG+LR fit seconds: 2.5


B2b HOG+GB fit seconds: 78.0


{('B1 Otsu+CCA', 'val'): '6.4 ms', ('B1 Otsu+CCA', 'test'): '6.1 ms', ('B2a HOG+LR', 'val'): '85.7 ms', ('B2b HOG+GB', 'val'): '87.0 ms', ('B2a HOG+LR', 'test'): '82.3 ms', ('B2b HOG+GB', 'test'): '86.2 ms'}


### 4a. B2 分類器選擇（只看 Val）

In [8]:
def val_ap(name):
    pm, gm = wb.match(preds[(name, "val")], veh, dc, ids["val"]); return wb.average_precision(pm, len(gm))
sel = {n: round(val_ap(n), 4) for n in clfs}
print("Val AP@0.5:", sel)
B2_NAME = max(sel, key=sel.get)
print("Selected Baseline 2 =", B2_NAME, "(selected on Validation only)")

Val AP@0.5: {'B2a HOG+LR': 0.163, 'B2b HOG+GB': 0.3242}
Selected Baseline 2 = B2b HOG+GB (selected on Validation only)


## 5. Candidate｜YOLO11n

In [9]:
from ultralytics import YOLO
import torch
if TRAIN_YOLO:
    # 建 YOLO 格式資料（Car+OtherVehicle → class 0；Train 全部含車影格 + 300 張隨機無車影格，seed 42）
    ydir = DATA_DIR / "yolo"
    for s in ["train", "val", "test"]:
        (ydir / "images" / s).mkdir(parents=True, exist_ok=True); (ydir / "labels" / s).mkdir(parents=True, exist_ok=True)
    pos = set(veh.image_id); tr = im[im.split == "train"]
    use_train = set(tr[tr.id.isin(pos)].id) | set(tr[~tr.id.isin(pos)].sample(300, random_state=RANDOM_STATE).id)
    import shutil
    for r in im.itertuples():
        if r.split == "train" and r.id not in use_train: continue
        shutil.copy(DATA_DIR / "images" / r.filename, ydir / "images" / r.split / r.filename)
        lab = veh[veh.image_id == r.id]
        (ydir / "labels" / r.split / r.filename.replace(".jpg", ".txt")).write_text("\n".join(
            f"0 {(a.x+a.w/2)/640:.6f} {(a.y+a.h/2)/512:.6f} {a.w/640:.6f} {a.h/512:.6f}" for a in lab.itertuples()))
    (ydir / "data.yaml").write_text(f"path: {ydir.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: vehicle\n")
    YOLO("yolo11n.pt").train(data=str(ydir / "data.yaml"), epochs=20, imgsz=512, batch=16, device="cpu", seed=RANDOM_STATE,
                             deterministic=True, project=str(ROOT / "runs"), name="yolo11n_v02", exist_ok=True, val=False, plots=False)
    YOLO_WEIGHTS.parent.mkdir(exist_ok=True)
    shutil.copy(ROOT / "runs" / "yolo11n_v02" / "weights" / "last.pt", YOLO_WEIGHTS)

torch.set_num_threads(2)
model = YOLO(str(YOLO_WEIGHTS))
for s in ["val", "test"]:
    rows, t_total = [], 0.0
    model.predict(str(DATA_DIR / "images" / fname[ids[s][0]]), imgsz=512, device="cpu", verbose=False)   # warm-up
    for iid in ids[s]:
        g = gray(iid); t0 = time.perf_counter()
        r = model.predict(cv2.cvtColor(g, cv2.COLOR_GRAY2BGR), imgsz=512, conf=0.001, iou=0.45, device="cpu", verbose=False, max_det=300)[0]
        t_total += time.perf_counter() - t0
        for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist()):
            rows.append(dict(image_id=iid, x=x1, y=y1, w=x2 - x1, h=y2 - y1, score=c))
    preds[("C YOLO11n", s)] = pd.DataFrame(rows); runtime[("C YOLO11n", s)] = 1000 * t_total / len(ids[s])
print({k: f"{v:.1f} ms" for k, v in runtime.items() if k[0] == "C YOLO11n"})

{('C YOLO11n', 'val'): '57.5 ms', ('C YOLO11n', 'test'): '56.9 ms'}


## 6. Results（Locked Test set）

In [10]:
METHODS = [("B1 Otsu+CCA", "Low", "W2 current / naive baseline"),
           (B2_NAME, "Low-Mid", "Credible simple baseline (classifier chosen on Val)"),
           ("C YOLO11n", "Mid", "Candidate; COCO-pretrained, fine-tuned 20 epochs on Train")]
results, matched = [], {}
for name, cx, note in METHODS:
    pm_v, gm_v = wb.match(preds[(name, "val")], veh, dc, ids["val"])
    t = wb.best_f1_threshold(pm_v, gm_v, len(ids["val"]))
    pm_t, gm_t = wb.match(preds[(name, "test")], veh, dc, ids["test"])
    r = wb.at_threshold(pm_t, gm_t, len(ids["test"]), t)
    gm_t["detected"] = gm_t.matched_score >= t
    pm_t["kept"] = (pm_t.score >= t) & ~pm_t.ignored
    matched[name] = (pm_t, gm_t, t)
    results.append({"method": name, "dataset_information": "640x512 IR pixels only; date split v0.2",
                    "val_AP50": wb.average_precision(pm_v, len(gm_v)),
                    "test_AP50 (primary)": wb.average_precision(pm_t, len(gm_t)),
                    "test_miss_rate (failure)": r["miss_rate"], "test_recall": r["recall"], "test_precision": r["precision"],
                    "false_alarms_per_100_frames": r["fp_per_100_frames"], "val_threshold": t,
                    "runtime_ms_per_frame": runtime[(name, "test")], "complexity": cx, "note": note})
results_df = pd.DataFrame(results)
results_df.round(4)

,method,dataset_information,val_AP50,test_AP50 (primary),test_miss_rate (failure),test_recall,test_precision,false_alarms_per_100_frames,val_threshold,runtime_ms_per_frame,complexity,note
0,B1 Otsu+CCA,640x512 IR pixels only; date split v0.2,0.0004,0.0081,0.9534,0.0466,0.0703,244.4268,1.3983,6.0571,Low,W2 current / naive baseline
1,B2b HOG+GB,640x512 IR pixels only; date split v0.2,0.3242,0.3637,0.5496,0.4504,0.4698,201.4331,0.8299,86.2004,Low-Mid,Credible simple baseline (classifier chosen on...
2,C YOLO11n,640x512 IR pixels only; date split v0.2,0.7186,0.7832,0.2929,0.7071,0.8314,56.8471,0.2504,56.8572,Mid,"Candidate; COCO-pretrained, fine-tuned 20 epoc..."


In [11]:
sub = []
for name, (pm_t, gm_t, t) in matched.items():
    for col in ["scale", "daynight", "altitude_m", "angle_deg", "cold"]:
        for k, v in gm_t.groupby(col, observed=True).detected.agg(["mean", "size"]).iterrows():
            sub.append(dict(method=name, group=col, value=str(k), recall=v["mean"], n_gt=int(v["size"])))
subgroup_df = pd.DataFrame(sub)
subgroup_df.pivot_table(index=["group", "value", "n_gt"], columns="method", values="recall").round(3)

method                   B1 Otsu+CCA  B2b HOG+GB  C YOLO11n
group      value   n_gt                                    
altitude_m 100     655         0.005       0.521      0.753
           110     571         0.007       0.282      0.718
           120     574         0.150       0.629      0.779
           130     381         0.031       0.323      0.491
           80      148         0.061       0.439      0.797
           90      160         0.012       0.438      0.656
angle_deg  30      610         0.018       0.159      0.405
           40      508         0.030       0.376      0.719
           50      232         0.065       0.371      0.746
           60      318         0.063       0.601      0.896
           70      249         0.080       0.715      0.908
           80      239         0.063       0.632      0.795
           90      333         0.060       0.682      0.823
cold       False   1053        0.108       0.506      0.758
           True    1436        0.001       0.409      0.670
daynight   day     1357        0.063       0.492      0.752
           night   1132        0.027       0.400      0.653
scale      medium+ 1802        0.054       0.546      0.774
           small   674         0.025       0.203      0.539
           tiny    13          0.077       0.000      0.154

### 6a. 穩定性：Test 兩個日期分開算 AP@0.5

In [12]:
stab = []
for name in matched:
    for d in sorted(im.loc[im.split == "test", "date_captured"].unique()):
        d_ids = list(im[(im.split == "test") & (im.date_captured == d)].id)
        pm_d, gm_d = wb.match(preds[(name, "test")], veh, dc, d_ids)
        stab.append(dict(method=name, date=d, frames=len(d_ids), n_gt=len(gm_d), AP50=round(wb.average_precision(pm_d, len(gm_d)), 4)))
stab_df = pd.DataFrame(stab); stab_df.pivot(index="method", columns="date", values="AP50")

date,20210121,20210123
method,,
B1 Otsu+CCA,0.0044,0.0165
B2b HOG+GB,0.3287,0.4565
C YOLO11n,0.8097,0.7295


## 7. Failure Analysis

In [13]:
def fp_merge_share(pm_t):
    """FP boxes (kept) that cover ≥ 2 GT vehicles (IoA ≥ 0.5 of each GT) → merged adjacent cars."""
    fp = pm_t[pm_t.kept & (pm_t.tp == 0)]; n = 0
    gv = veh[veh.split == "test"]
    for iid, g in fp.groupby("image_id"):
        G = gv[gv.image_id == iid]
        if len(G) < 2: continue
        cov = wb.ioa_pred(G[["x", "y", "w", "h"]].values.astype(float), g[["x", "y", "w", "h"]].values.astype(float))
        n += int(((cov >= 0.5).sum(0) >= 2).sum())
    return n, len(fp)

for name, (pm_t, gm_t, t) in matched.items():
    m, nfp = fp_merge_share(pm_t)
    missed = gm_t[~gm_t.detected]
    print(f"{name}: missed={len(missed)}  | missed that are cold: {missed.cold.mean()*100:.1f}%  | FP={nfp}, FP covering ≥2 cars: {m}")

B1 Otsu+CCA: missed=2373  | missed that are cold: 60.4%  | FP=1535, FP covering ≥2 cars: 8


B2b HOG+GB: missed=1368  | missed that are cold: 62.0%  | FP=1265, FP covering ≥2 cars: 70
C YOLO11n: missed=729  | missed that are cold: 65.0%  | FP=357, FP covering ≥2 cars: 34


In [14]:
FAIL_DIR = OUT / "week04_failure_cases"; FAIL_DIR.mkdir(exist_ok=True)
def frame_stats(name):
    pm_t, gm_t, t = matched[name]
    a = gm_t.groupby("image_id").agg(n_gt=("detected", "size"), n_hit=("detected", "sum"), cold=("cold", "mean"))
    a["n_fp"] = pm_t[pm_t.kept & (pm_t.tp == 0)].groupby("image_id").size().reindex(a.index).fillna(0).astype(int)
    a = a.join(im.set_index("id")[["filename", "daynight", "altitude_m"]])
    return a

def draw(name, iid, path):
    pm_t, gm_t, t = matched[name]
    img = cv2.cvtColor(gray(iid), cv2.COLOR_GRAY2BGR)
    for a in gm_t[gm_t.image_id == iid].itertuples():
        cv2.rectangle(img, (int(a.x), int(a.y)), (int(a.x + a.w), int(a.y + a.h)), (0, 200, 0) if a.detected else (0, 0, 255), 2)
    for p in pm_t[(pm_t.image_id == iid) & pm_t.kept & (pm_t.tp == 0)].itertuples():
        cv2.rectangle(img, (int(p.x), int(p.y)), (int(p.x + p.w), int(p.y + p.h)), (0, 215, 255), 1)
    cv2.putText(img, f"{name} | green=hit red=miss yellow=FP", (5, 18), 0, 0.5, (255, 255, 255), 1)
    cv2.imwrite(str(path), img)

cases = []
# Case 1 — B1: busy frame with mostly cold vehicles, B1 finds (almost) none
s1 = frame_stats("B1 Otsu+CCA"); c1 = s1[(s1.n_gt >= 8)].sort_values(["n_hit", "cold", "n_gt"], ascending=[True, False, False]).index[0]
# Case 2 — B2: frame with the most FP boxes that swallow ≥2 adjacent vehicles
pm2, gm2, _ = matched[B2_NAME]; best, c2 = -1, None
for iid, g in pm2[pm2.kept & (pm2.tp == 0)].groupby("image_id"):
    G = veh[(veh.image_id == iid)]
    if len(G) < 2: continue
    k = int(((wb.ioa_pred(G[["x","y","w","h"]].values.astype(float), g[["x","y","w","h"]].values.astype(float)) >= 0.5).sum(0) >= 2).sum())
    if k > best: best, c2 = k, iid
# Case 3 — YOLO: frame with most missed vehicles
s3 = frame_stats("C YOLO11n"); s3["n_miss"] = s3.n_gt - s3.n_hit; c3 = s3.sort_values(["n_miss", "n_gt"], ascending=False).index[0]
# Case 4 — YOLO: frame with most false alarms
c4 = s3.sort_values("n_fp", ascending=False).index[0]
for k, (name, iid) in enumerate([("B1 Otsu+CCA", c1), (B2_NAME, c2), ("C YOLO11n", c3), ("C YOLO11n", c4)], 1):
    path = FAIL_DIR / f"case{k}_{name.split()[0]}_{fname[iid]}"; draw(name, iid, path)
    st = frame_stats(name).loc[iid]
    cases.append(dict(case=k, method=name, frame=fname[iid], daynight=st.daynight, altitude_m=st.altitude_m,
                      gt_vehicles=int(st.n_gt), detected=int(st.n_hit), false_alarms=int(st.n_fp), cold_share=round(st.cold, 2), image=path.name))
cases_df = pd.DataFrame(cases); cases_df

,case,method,frame,daynight,altitude_m,gt_vehicles,detected,false_alarms,cold_share,image
0,1,B1 Otsu+CCA,1_110_30_0_09028.jpg,night,110,13,0,0,1.00,case1_B1_1_110_30_0_09028.jpg
1,2,B2b HOG+GB,1_130_30_0_09965.jpg,night,130,9,1,3,0.89,case2_B2b_1_130_30_0_09965.jpg
2,3,C YOLO11n,0_100_30_0_08087.jpg,day,100,18,3,1,0.33,case3_C_0_100_30_0_08087.jpg
3,4,C YOLO11n,0_110_30_0_08339.jpg,day,110,28,25,11,0.32,case4_C_0_110_30_0_08339.jpg


In [15]:
pm3, gm3, _ = matched["C YOLO11n"]
print("Case 3 missed boxes (w×h, scale, cold):")
print(gm3[(gm3.image_id == c3) & ~gm3.detected][["x", "y", "w", "h", "scale", "cold", "contrast"]].round(1).to_string())
print("\nCase 4 false alarms (score):")
print(pm3[(pm3.image_id == c4) & pm3.kept & (pm3.tp == 0)][["x", "y", "w", "h", "score"]].round(2).to_string())

Case 3 missed boxes (w×h, scale, cold):
        x    y   w   h    scale   cold  contrast
5647  377  248  30  36  medium+   True      -1.6
5648  405  250  31  42  medium+  False      11.5
5650  436   43  25  24    small   True     -18.9
5651  138   51  15  22    small  False      67.3
5652  146  212  21  35    small   True      -1.6
5653  480   89  25  27    small  False       1.3
5654  581  121  18  27    small  False      13.8
5655  517  112  21  27    small   True     -11.7
5656  537   33  19  24    small   True      -3.6
5657  565   39  15  22    small  False      70.1
5658  624  125  15  30    small  False      22.6
5659  593   40  19  20    small  False      25.8
5660  584   55  18  28    small  False       1.0
5662  377   91  32  31    small  False       5.7
5664  629   63  11  21     tiny  False       4.4

Case 4 false alarms (score):
            x       y      w      h  score
10211  568.40  363.19  20.64  27.45   0.80
10221  591.21  385.46  36.06  39.17   0.70
10225  539.34  27

## 8. Save Evidence / Run Record

In [16]:
results_df.round(4).to_csv(OUT / "week04_results.csv", index=False, encoding="utf-8-sig")
subgroup_df.round(4).to_csv(OUT / "week04_subgroup_recall.csv", index=False, encoding="utf-8-sig")
stab_df.to_csv(OUT / "week04_test_by_date.csv", index=False, encoding="utf-8-sig")
cases_df.to_csv(OUT / "week04_failure_cases.csv", index=False, encoding="utf-8-sig")
run = pd.DataFrame([{"run_name": f"w4_{r['method'].split()[0]}_{DATASET_VERSION}", "method": r["method"], "data_version": DATASET_VERSION,
                     "split": "date v0.2 (test=20210121,20210123)", "seed": RANDOM_STATE, "threshold": round(r["val_threshold"], 4),
                     "feature_set": "IR pixels only", "test_AP50": round(r["test_AP50 (primary)"], 4),
                     "test_miss_rate": round(r["test_miss_rate (failure)"], 4), "ms_per_frame": round(r["runtime_ms_per_frame"], 1)} for r in results])
run.to_csv(OUT / "week04_run_records.csv", index=False, encoding="utf-8-sig")
print(run.to_string(index=False))

                  run_name      method        data_version                              split  seed  threshold    feature_set  test_AP50  test_miss_rate  ms_per_frame
 w4_B1_hituav-vehicle-v0.2 B1 Otsu+CCA hituav-vehicle-v0.2 date v0.2 (test=20210121,20210123)    42     1.3983 IR pixels only     0.0081          0.9534           6.1
w4_B2b_hituav-vehicle-v0.2  B2b HOG+GB hituav-vehicle-v0.2 date v0.2 (test=20210121,20210123)    42     0.8299 IR pixels only     0.3637          0.5496          86.2
  w4_C_hituav-vehicle-v0.2   C YOLO11n hituav-vehicle-v0.2 date v0.2 (test=20210121,20210123)    42     0.2504 IR pixels only     0.7832          0.2929          56.9
